In [2]:
import os
import json
import math
import random
import copy
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import timm
import wandb
from torchmetrics.classification import MulticlassAveragePrecision

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Mounted at /content/drive


In [3]:
BASE_PATH = Path('/content/drive/My Drive/Plant Disease')
PREPARED_DIR = BASE_PATH / 'prepared'
CKPT_DIR = BASE_PATH / 'checkpoints'   # one subfolder per experiment
CKPT_DIR.mkdir(exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
SEED = 42
WANDB_PROJECT = 'plant disease classification'


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
print("Ready.")

Ready.


In [4]:
train_df = pd.read_csv(PREPARED_DIR / 'metadata_train.csv')
val_df   = pd.read_csv(PREPARED_DIR / 'metadata_val.csv')

with open(PREPARED_DIR / 'label_map.json', encoding='utf-8') as f:
    label_map = json.load(f)

class_to_id: Dict[str, int] = label_map['class_to_id']
id_to_class: Dict[int, str] = {int(k): v for k, v in label_map['id_to_class'].items()}
NUM_CLASSES = len(class_to_id)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Classes: {NUM_CLASSES}")

counts  = train_df['disease_label'].value_counts()
weights = [1.0 / counts[id_to_class[i]] for i in range(NUM_CLASSES)]
TRAIN_SAMPLE_WEIGHTS = torch.tensor(
    [weights[class_to_id[label]] for label in train_df['disease_label']],
    dtype=torch.float,
)
print("Sample weight range:", TRAIN_SAMPLE_WEIGHTS.min().item(), "→", TRAIN_SAMPLE_WEIGHTS.max().item())

Train: 7783  |  Val: 390  |  Classes: 39
Sample weight range: 0.0011547344038262963 → 0.011111111380159855


In [5]:
class PlantDiseaseDataset(Dataset):
    """
    return_pil=False  →  returns (tensor, label)  — used during training
    return_pil=True   →  returns (PIL image, label) — used for TTA at eval
    """

    def __init__(self, df, class_to_id, transform=None, return_pil=False):
        self.df = df.reset_index(drop=True)
        self.class_to_id = class_to_id
        self.transform = transform
        self.return_pil = return_pil

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        label = self.class_to_id[row['disease_label']]
        if self.return_pil:
            return image, label
        return self.transform(image), label


def build_loaders(train_transform, val_transform, batch_size, num_workers=2):
    train_ds = PlantDiseaseDataset(train_df, class_to_id, transform=train_transform)
    val_ds = PlantDiseaseDataset(val_df, class_to_id, transform=val_transform)

    sampler = WeightedRandomSampler(
        weights=TRAIN_SAMPLE_WEIGHTS,
        num_samples=len(train_ds),
        replacement=True,
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader


print("Dataset class ready.")

Dataset class ready.


In [6]:
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, backbone_name, num_classes, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.backbone.num_features, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        for p in self.head.parameters():     p.requires_grad = True

    def unfreeze_backbone(self):
        for p in self.parameters(): p.requires_grad = True

    def get_param_groups(self, lr_backbone, lr_head, weight_decay):
        return [
            {'params': self.backbone.parameters(), 'lr': lr_backbone, 'weight_decay': weight_decay},
            {'params': self.head.parameters(), 'lr': lr_head, 'weight_decay': weight_decay},
        ]


print("Model class ready.")

Model class ready.


In [7]:
def mixup(x, y, alpha=0.2):
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, _, H, W = x.shape
    idx  = torch.randperm(B, device=x.device)
    ch   = int(H * math.sqrt(1 - lam))
    cw   = int(W * math.sqrt(1 - lam))
    cx   = random.randint(0, W)
    cy   = random.randint(0, H)
    x1, y1 = max(cx - cw // 2, 0), max(cy - ch // 2, 0)
    x2, y2 = min(cx + cw // 2, W), min(cy + ch // 2, H)
    out  = x.clone()
    out[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam  = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return out, y, y[idx], lam


def apply_mix(x, y, mixup_alpha=0.2, cutmix_alpha=1.0, mixup_prob=0.5):
    if random.random() < mixup_prob:
        return mixup(x, y, mixup_alpha)
    return cutmix(x, y, cutmix_alpha)


def mix_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


print("MixUp/CutMix ready.")

MixUp/CutMix ready.


In [8]:
class EarlyStopping:
    def __init__(self, patience: int, path: Path):
        self.patience   = patience
        self.path       = path
        self.best_score = -1.0
        self.counter    = 0
        self.best_state = None

    def step(self, score: float, model: nn.Module) -> bool:
        """Returns True when training should stop."""
        if score > self.best_score:
            self.best_score = score
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
            torch.save(self.best_state, self.path)
            print(f"  ✓ mAP {score:.4f} — saved")
            return False
        self.counter += 1
        print(f"  No improvement ({self.counter}/{self.patience})")
        return self.counter >= self.patience

    def restore(self, model: nn.Module):
        model.load_state_dict(self.best_state)
        print(f"Restored best (mAP={self.best_score:.4f})")


def train_epoch(model, loader, optimizer, criterion, accum_steps=1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs, y_a, y_b, lam = apply_mix(imgs, labels)

        logits = model(imgs)
        loss   = mix_loss(criterion, logits, y_a, y_b, lam) / accum_steps
        loss.backward()

        if (i + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps   # unscale for logging
        correct    += (logits.argmax(1) == y_a).sum().item()
        total      += labels.size(0)

    return {'train/loss': total_loss / len(loader),
            'train/acc':  correct / total}


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss  = 0.0
    all_logits, all_labels = [], []
    map_fn = MulticlassAveragePrecision(num_classes=NUM_CLASSES, average='macro').to(DEVICE)

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        all_logits.append(logits)
        all_labels.append(labels)

    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)
    probs = F.softmax(all_logits, dim=1)

    return {
        'val/loss': total_loss / len(loader),
        'val/acc':  (all_logits.argmax(1) == all_labels).float().mean().item(),
        'val/mAP':  map_fn(probs, all_labels).item(),
    }


@torch.no_grad()
def tta_eval(model, val_transform, tta_transform, n_views=5):
    """
    For every val image:
      - view 0 : clean centre-crop  (val_transform)
      - views 1-4 : random augments (tta_transform)
    Average softmax probs across all views, then compute mAP.
    """
    model.eval()
    map_fn   = MulticlassAveragePrecision(num_classes=NUM_CLASSES, average='macro').to(DEVICE)
    pil_ds   = PlantDiseaseDataset(val_df, class_to_id, return_pil=True)
    # collate_fn=lambda x: x keeps items as a plain list — needed because
    # PIL images can't be stacked by the default collate
    pil_loader = DataLoader(pil_ds, batch_size=1, shuffle=False,
                            collate_fn=lambda x: x)
    all_probs, all_labels = [], []

    for batch in pil_loader:
        pil_img, label = batch[0]

        # Build n_views tensors: first is always clean, rest are augmented
        views = [val_transform(pil_img).unsqueeze(0).to(DEVICE)]
        for _ in range(n_views - 1):
            views.append(tta_transform(pil_img).unsqueeze(0).to(DEVICE))

        # Forward all views in one batch → average probabilities
        logits    = model(torch.cat(views))             # (n_views, C)
        avg_probs = F.softmax(logits, dim=1).mean(0, keepdim=True)  # (1, C)

        all_probs.append(avg_probs)
        all_labels.append(torch.tensor([label]).to(DEVICE))

    return map_fn(torch.cat(all_probs), torch.cat(all_labels)).item()


@torch.no_grad()
def per_class_eval(model, loader):
    model.eval()
    map_fn = MulticlassAveragePrecision(num_classes=NUM_CLASSES, average=None).to(DEVICE)
    all_logits, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        all_logits.append(model(imgs))
        all_labels.append(labels)
    probs = F.softmax(torch.cat(all_logits), dim=1)
    ap    = map_fn(probs, torch.cat(all_labels)).cpu().numpy()
    return pd.DataFrame({
        'disease':       [id_to_class[i] for i in range(NUM_CLASSES)],
        'avg_precision': ap,
    }).sort_values('avg_precision', ascending=False).reset_index(drop=True)


print("All utilities ready.")

All utilities ready.


In [9]:
def run_experiment(cfg: dict) -> dict:
    """
    Full training pipeline from scratch for one backbone config.

    Required cfg keys:
        name, backbone, img_size, batch_size, dropout,
        phase1_epochs, phase1_lr,
        phase2_epochs, phase2_lr_head, phase2_lr_backbone,
        weight_decay, label_smoothing, early_stop_patience,
        accumulation_steps

    Returns:
        dict with name, best_map, tta_map, per_class_df, checkpoint_path
    """
    name        = cfg['name']
    ckpt_path   = CKPT_DIR / f'plantdiseaseclassifier_{name}_best.pt'
    accum       = cfg.get('accumulation_steps', 1)
    eff_batch   = cfg['batch_size'] * accum

    print(f"\n{'='*65}")
    print(f"  {name}")
    print(f"  backbone={cfg['backbone']}  img={cfg['img_size']}px  "
          f"batch={cfg['batch_size']}x{accum}={eff_batch}")
    print(f"{'='*65}")

    seed_everything(SEED)

    sz = cfg['img_size']
    train_tf = T.Compose([
        T.RandomResizedCrop(sz, scale=(0.5, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        T.RandomRotation(30),
        T.RandAugment(num_ops=2, magnitude=9),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_tf = T.Compose([
        T.Resize(int(sz * 1.14)),
        T.CenterCrop(sz),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_loader, val_loader = build_loaders(
        train_tf, val_tf, cfg['batch_size']
    )

    model = PlantDiseaseClassifier(
        backbone_name=cfg['backbone'],
        num_classes=NUM_CLASSES,
        dropout=cfg.get('dropout', 0.3),
        pretrained=True,                  # start from ImageNet weights
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Params: {n_params:.2f}M")

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.get('label_smoothing', 0.1))

    wandb.init(project=WANDB_PROJECT, name=name, config=cfg, reinit=True)

    print(f"\n--- Phase 1: frozen backbone ({cfg['phase1_epochs']} epochs) ---")
    model.freeze_backbone()
    head_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable: {head_params/1e3:.1f}K (head only)")

    opt1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['phase1_lr'],
        weight_decay=cfg.get('weight_decay', 1e-4),
    )
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=cfg['phase1_epochs'])
    es1  = EarlyStopping(patience=cfg.get('early_stop_patience', 10), path=ckpt_path)

    for ep in range(1, cfg['phase1_epochs'] + 1):
        tm = train_epoch(model, train_loader, opt1, criterion, accum)
        vm = val_epoch(model, val_loader, criterion)
        sch1.step()
        wandb.log({**tm, **vm, 'epoch': ep, 'phase': 1})
        print(f"  P1 E{ep:02d} | loss {tm['train/loss']:.4f} acc {tm['train/acc']:.3f}"
              f" | val loss {vm['val/loss']:.4f} acc {vm['val/acc']:.3f} mAP {vm['val/mAP']:.4f}")
        if es1.step(vm['val/mAP'], model):
            print("  Early stop — Phase 1")
            break

    es1.restore(model)
    print(f"  Phase 1 best mAP: {es1.best_score:.4f}")

    print(f"\n--- Phase 2: full fine-tune ({cfg['phase2_epochs']} epochs) ---")
    model.unfreeze_backbone()
    all_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Trainable: {all_params:.2f}M (full model)")

    opt2 = torch.optim.AdamW(
        model.get_param_groups(
            lr_backbone=cfg['phase2_lr_backbone'],
            lr_head=cfg['phase2_lr_head'],
            weight_decay=cfg.get('weight_decay', 1e-4),
        )
    )
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=cfg['phase2_epochs'])
    es2  = EarlyStopping(patience=cfg.get('early_stop_patience', 10), path=ckpt_path)

    for ep in range(1, cfg['phase2_epochs'] + 1):
        tm = train_epoch(model, train_loader, opt2, criterion, accum)
        vm = val_epoch(model, val_loader, criterion)
        sch2.step()
        wandb.log({**tm, **vm, 'epoch': ep + cfg['phase1_epochs'], 'phase': 2})
        print(f"  P2 E{ep:02d} | loss {tm['train/loss']:.4f} acc {tm['train/acc']:.3f}"
              f" | val loss {vm['val/loss']:.4f} acc {vm['val/acc']:.3f} mAP {vm['val/mAP']:.4f}")
        if es2.step(vm['val/mAP'], model):
            print("  Early stop — Phase 2")
            break

    es2.restore(model)
    best_map = es2.best_score
    print(f"  Phase 2 best mAP: {best_map:.4f}")

    print("\n  Running TTA (5 views)...")
    tta_map = tta_eval(model, val_tf, train_tf, n_views=5)
    print(f"  TTA mAP: {tta_map:.4f}  (plain: {best_map:.4f},  gain: +{tta_map-best_map:.4f})")

    pc_df = per_class_eval(model, val_loader)
    print("\n  Per-class AP:")
    print(pc_df.to_string(index=False))

    bundle_path = CKPT_DIR / f'plantdiseaseclassifier_{name}_bundle.pt'
    torch.save({
        'state_dict':    model.state_dict(),
        'backbone':      cfg['backbone'],
        'num_classes':   NUM_CLASSES,
        'dropout':       0.0,            # disabled at inference
        'img_size':      cfg['img_size'],
        'class_to_id':   class_to_id,
        'id_to_class':   id_to_class,
        'val_mAP':       best_map,
        'tta_mAP':       tta_map,
        'imagenet_mean': IMAGENET_MEAN,
        'imagenet_std':  IMAGENET_STD,
    }, bundle_path)
    print(f"\n  Bundle saved → {bundle_path.name}")

    wandb.log({
        'best_val_mAP': best_map,
        'tta_mAP':      tta_map,
        'per_class_AP': wandb.Table(dataframe=pc_df),
    })
    wandb.run.summary.update({'best_val_mAP': best_map, 'tta_mAP': tta_map})
    wandb.finish()

    return {
        'name':            name,
        'backbone':        cfg['backbone'],
        'img_size':        cfg['img_size'],
        'best_map':        best_map,
        'tta_map':         tta_map,
        'per_class_df':    pc_df,
        'checkpoint_path': ckpt_path,
        'bundle_path':     bundle_path,
        'model':           model,
    }


print("run_experiment() ready.")

run_experiment() ready.


In [10]:
result_b2 = run_experiment({
    'name':                'efficientnet_b2',
    'backbone':            'efficientnet_b2',
    'img_size':            260,
    'batch_size':          48,       # smaller than B0 to fit 260px on T4
    'dropout':             0.3,

    'phase1_epochs':       10,
    'phase1_lr':           1e-3,

    'phase2_epochs':       50,
    'phase2_lr_head':      1e-4,
    'phase2_lr_backbone':  1e-5,

    'weight_decay':        1e-4,
    'label_smoothing':     0.1,
    'early_stop_patience': 10,
    'accumulation_steps':  1,
})


  efficientnet_b2
  backbone=efficientnet_b2  img=260px  batch=48x1=48


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

  Params: 7.76M


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mane-musayelyan (mane-musayelyan-yerevan-state-university-ysu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



--- Phase 1: frozen backbone (10 epochs) ---
  Trainable: 55.0K (head only)
  P1 E01 | loss 3.1982 acc 0.174 | val loss 2.6551 acc 0.431 mAP 0.4922
  ✓ mAP 0.4922 — saved
  P1 E02 | loss 2.7941 acc 0.217 | val loss 2.3499 acc 0.497 mAP 0.5656
  ✓ mAP 0.5656 — saved
  P1 E03 | loss 2.6750 acc 0.304 | val loss 2.2745 acc 0.526 mAP 0.5711
  ✓ mAP 0.5711 — saved
  P1 E04 | loss 2.6200 acc 0.287 | val loss 2.1702 acc 0.569 mAP 0.6039
  ✓ mAP 0.6039 — saved
  P1 E05 | loss 2.6096 acc 0.274 | val loss 2.1338 acc 0.572 mAP 0.6221
  ✓ mAP 0.6221 — saved
  P1 E06 | loss 2.5860 acc 0.299 | val loss 2.1307 acc 0.572 mAP 0.6165
  No improvement (1/10)
  P1 E07 | loss 2.5398 acc 0.305 | val loss 2.1003 acc 0.574 mAP 0.6198
  No improvement (2/10)
  P1 E08 | loss 2.5185 acc 0.304 | val loss 2.0935 acc 0.569 mAP 0.6223
  ✓ mAP 0.6223 — saved
  P1 E09 | loss 2.5548 acc 0.308 | val loss 2.0672 acc 0.595 mAP 0.6323
  ✓ mAP 0.6323 — saved
  P1 E10 | loss 2.5555 acc 0.318 | val loss 2.0790 acc 0.582 mAP 0

best_val_mAP,▁
epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██
phase,▁▁▁▁▁▁██████████████████████████████████
train/acc,▁▂▄▃▃▄▄▄▄▅▅▅▄▆▆▆▆▆▆▇▇▇▆▆▆██▇█▇▇▇█▇▇▇▆▇▇▆
train/loss,█▆▅▅▅▄▄▄▄▃▃▂▃▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▂▁▁▁▁▁
tta_mAP,▁
val/acc,▁▃▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇█▇██████████████
val/loss,█▆▆▅▅▅▅▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/mAP,▁▁▂▃▂▃▃▃▄▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
best_val_mAP,0.82883
epoch,60
